In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
stg = ROOT / "data" / "staging"

bc = pd.read_csv(stg / "budget_categories.csv")
sr = pd.read_csv(stg / "spending_records.csv", parse_dates=["transaction_date"])

bc["period"] = pd.to_datetime(bc["year"].astype(str) + "-" + bc["month"].astype(str).str.zfill(2) + "-01")
print("households:", bc["user_profile_id"].nunique())
print("months:", bc["period"].min().date(), "to", bc["period"].max().date(), "| distinct:", bc["period"].nunique())
print("categories:", sorted(bc["category"].unique()))
print("approval_status:", bc["approval_status"].value_counts().to_dict())

per_hh = bc.groupby("user_profile_id")["period"].nunique()
print("months per household:", per_hh.describe()[["min", "50%", "max"]].to_dict())

print("anomaly share:", round(sr["is_anomaly"].mean(), 3), sr["anomaly_type"].value_counts().head().to_dict())

sr["period"] = sr["transaction_date"].dt.to_period("M").dt.to_timestamp()
agg = sr.groupby(["user_profile_id", "period", "category"])["amount_kes"].sum().reset_index()
m = bc.merge(agg, on=["user_profile_id", "period", "category"], how="left")
print("actual_kes equals summed spending (within 1 KES):", ((m["actual_kes"] - m["amount_kes"]).abs() < 1).mean().round(3))
print("budget rows with no spending records:", m["amount_kes"].isna().mean().round(3))

households: 500
months: 2025-04-01 to 2026-09-01 | distinct: 18
categories: ['Airtime and Data', 'Cash Advance', 'Crypto Purchase', 'Electronics Upgrade', 'Entertainment', 'Foreign Transfer', 'Gambling', 'Groceries', 'Household Help', 'Jewellery', 'Luxury Goods', 'Medical', 'Rent', 'Savings', 'School Fees', 'Transport', 'Utilities']
approval_status: {'APPROVED': 64019, 'PENDING': 16526}
months per household: {'min': 18.0, '50%': 18.0, 'max': 18.0}
anomaly share: 0.017 {'frequency_burst': 3185, 'duplicate_charge': 774, 'new_category': 411, 'amount_spike': 398, 'odd_hour': 391}
actual_kes equals summed spending (within 1 KES): 1.0
budget rows with no spending records: 0.0


In [2]:
import numpy as np

print("PENDING share by month:")
print(bc.groupby("period")["approval_status"].apply(lambda s: (s == "PENDING").mean()).round(2).to_string())

periods = sorted(bc["period"].unique())
hh = sorted(bc["user_profile_id"].unique())
wide = bc.pivot_table(index=["user_profile_id", "period"], columns="category", values="actual_kes", aggfunc="sum")
wide = wide.reindex(pd.MultiIndex.from_product([hh, periods], names=["user_profile_id", "period"])).fillna(0)
cats = list(wide.columns)
X = wide.to_numpy().reshape(len(hh), len(periods), len(cats))
print("array shape (households, months, categories):", X.shape)

def forecast(X, t, method):
    h = X[:, :t, :]
    if method == "last_month": return h[:, -1, :]
    if method == "mean_3m": return h[:, -3:, :].mean(axis=1)
    if method == "median_3m": return np.median(h[:, -3:, :], axis=1)
    if method == "median_6m": return np.median(h[:, -6:, :], axis=1)
    if method == "same_month_last_year": return X[:, t - 12, :]

rows = []
for method in ["last_month", "mean_3m", "median_3m", "median_6m", "same_month_last_year"]:
    ae_total, ae_cat, actual_total = [], [], []
    for t in range(12, 18):
        f, a = forecast(X, t, method), X[:, t, :]
        ae_total.append(np.abs(f.sum(axis=1) - a.sum(axis=1)))
        ae_cat.append(np.abs(f - a).sum(axis=1))
        actual_total.append(a.sum(axis=1))
    ae_total, ae_cat, actual_total = map(np.concatenate, (ae_total, ae_cat, actual_total))
    rows.append({"method": method,
                 "total_MAE_KES": round(ae_total.mean()),
                 "total_WAPE": round(ae_total.sum() / actual_total.sum(), 3),
                 "category_WAPE": round(ae_cat.sum() / actual_total.sum(), 3)})

print(pd.DataFrame(rows).to_string(index=False))
print("average actual total per household-month:", round(X[:, 12:, :].sum(axis=2).mean()))

PENDING share by month:
period
2025-04-01    0.20
2025-05-01    0.21
2025-06-01    0.21
2025-07-01    0.21
2025-08-01    0.21
2025-09-01    0.21
2025-10-01    0.20
2025-11-01    0.20
2025-12-01    0.20
2026-01-01    0.20
2026-02-01    0.19
2026-03-01    0.21
2026-04-01    0.20
2026-05-01    0.22
2026-06-01    0.20
2026-07-01    0.21
2026-08-01    0.21
2026-09-01    0.21
array shape (households, months, categories): (500, 18, 17)
              method  total_MAE_KES  total_WAPE  category_WAPE
          last_month          11621       0.156          0.350
             mean_3m           9674       0.130          0.298
           median_3m           9556       0.128          0.287
           median_6m           9024       0.121          0.266
same_month_last_year          12428       0.167          0.359
average actual total per household-month: 74433


In [3]:
def med(X, t, w): return np.median(X[:, max(0, t - w):t, :], axis=1)
def avg(X, t, w): return X[:, max(0, t - w):t, :].mean(axis=1)

rows = []
for name, fn, w in [("median_3m", med, 3), ("median_6m", med, 6), ("median_9m", med, 9),
                    ("median_12m", med, 12), ("mean_6m", avg, 6), ("mean_12m", avg, 12)]:
    ae_total, ae_cat, actual_total = [], [], []
    for t in range(12, 18):
        f, a = fn(X, t, w), X[:, t, :]
        ae_total.append(np.abs(f.sum(axis=1) - a.sum(axis=1)))
        ae_cat.append(np.abs(f - a).sum(axis=1))
        actual_total.append(a.sum(axis=1))
    ae_total, ae_cat, actual_total = map(np.concatenate, (ae_total, ae_cat, actual_total))
    rows.append({"method": name,
                 "total_MAE_KES": round(ae_total.mean()),
                 "total_WAPE": round(ae_total.sum() / actual_total.sum(), 3),
                 "category_WAPE": round(ae_cat.sum() / actual_total.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

    method  total_MAE_KES  total_WAPE  category_WAPE
 median_3m           9556       0.128          0.287
 median_6m           9024       0.121          0.266
 median_9m           9240       0.124          0.260
median_12m           9180       0.123          0.255
   mean_6m           8978       0.121          0.283
  mean_12m           8779       0.118          0.277


In [4]:
d = bc[bc["allocated_kes"] > 0].copy()
d["over"] = d["actual_kes"] > d["allocated_kes"]
d["ratio"] = d["actual_kes"] / d["allocated_kes"]

print("overspend rate by approval_status:")
print(d.groupby("approval_status")["over"].mean().round(2))
print("\noverspend rate by is_recurring:")
print(d.groupby("is_recurring")["over"].mean().round(2))
print("\noverspend rate by category:")
print(d.groupby("category")["over"].mean().round(2).sort_values().to_string())
print("\nactual / allocated ratio:")
print(d["ratio"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2))

overspend rate by approval_status:
approval_status
APPROVED    0.34
PENDING     0.35
Name: over, dtype: float64

overspend rate by is_recurring:
is_recurring
f    0.34
t    0.34
Name: over, dtype: float64

overspend rate by category:
category
Medical             0.33
Savings             0.33
Groceries           0.34
School Fees         0.34
Transport           0.34
Airtime and Data    0.35
Entertainment       0.35
Household Help      0.35
Utilities           0.35
Rent                0.37

actual / allocated ratio:
count    80134.00
mean         0.99
std          0.33
min          0.56
10%          0.73
25%          0.80
50%          0.91
75%          1.11
90%          1.36
max         10.95
Name: ratio, dtype: float64


In [5]:
m = bc.groupby(["user_profile_id", "period"])[["actual_kes", "allocated_kes"]].sum().reset_index()
m = m[m["period"] >= "2026-04-01"]

err = (m["actual_kes"] - m["allocated_kes"]).abs()
print("total spend, forecasting with the month's budget:")
print("  MAE KES:", round(err.mean()), "| WAPE:", round(err.sum() / m["actual_kes"].sum(), 3))

rows = bc[bc["period"] >= "2026-04-01"]
cat_err = (rows["actual_kes"] - rows["allocated_kes"]).abs()
print("by category, forecasting with the month's budget:")
print("  category WAPE:", round(cat_err.sum() / rows["actual_kes"].sum(), 3))

total spend, forecasting with the month's budget:
  MAE KES: 7853 | WAPE: 0.106
by category, forecasting with the month's budget:
  category WAPE: 0.222


In [6]:
import numpy as np

periods = sorted(bc["period"].unique())
hh = sorted(bc["user_profile_id"].unique())
cats = sorted(bc["category"].unique())
idx = pd.MultiIndex.from_product([hh, periods], names=["user_profile_id", "period"])

def cube(col):
    w = bc.pivot_table(index=["user_profile_id", "period"], columns="category", values=col, aggfunc="sum")
    w = w.reindex(index=idx, columns=cats).fillna(0)
    return w.to_numpy().reshape(len(hh), len(periods), len(cats))

A, B = cube("actual_kes"), cube("allocated_kes")

rows = []
for w in [0, 0.25, 0.5, 0.75, 1.0]:
    ae_t, ae_c, act = [], [], []
    for t in range(12, 18):
        med = np.median(A[:, t - 12:t, :], axis=1)
        f = w * B[:, t, :] + (1 - w) * med
        a = A[:, t, :]
        ae_t.append(np.abs(f.sum(axis=1) - a.sum(axis=1)))
        ae_c.append(np.abs(f - a).sum(axis=1))
        act.append(a.sum(axis=1))
    ae_t, ae_c, act = map(np.concatenate, (ae_t, ae_c, act))
    rows.append({"budget_weight": w, "total_MAE": round(ae_t.mean()),
                 "total_WAPE": round(ae_t.sum() / act.sum(), 3),
                 "category_WAPE": round(ae_c.sum() / act.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

 budget_weight  total_MAE  total_WAPE  category_WAPE
          0.00       9180       0.123          0.255
          0.25       8302       0.112          0.241
          0.50       7788       0.105          0.230
          0.75       7635       0.103          0.222
          1.00       7853       0.106          0.222
